Notebook to demonstrate the non-dimensionalization of the robust-diagnostic planetary-geostrophic equations

twnh Jun '26

In [1]:
using Symbolics
using Latexify

@variables x y z

Dx = Differential(x)
Dy = Differential(y)
Dz = Differential(z)

@variables u(..) v(..) w(..) p(..) b(..) B(..)
@variables f(..) ν(..) κ(..)
@variables ρ₀ γ

u_dim = u(x,y,z)
v_dim = v(x,y,z)
w_dim = w(x,y,z)
p_dim = p(x,y,z)
b_dim = b(x,y,z)
B_dim = B(x,y,z)

f_dim = f(y)
ν_dim = ν(z)
κ_dim = κ(z) 

@variables u_nd(..) v_nd(..) w_nd(..) p_nd(..) b_nd(..) B_nd(..)
@variables f_nd(..) ν_nd(..) κ_nd(..)

û = u_nd(x,y,z)
v̂ = v_nd(x,y,z)
ŵ = w_nd(x,y,z)
p̂ = p_nd(x,y,z)
b̂ = b_nd(x,y,z)
B̂ = B_nd(x,y,z)

f̂ = f_nd(y)
ν̂ = ν_nd(z)
κ̂ = κ_nd(z) ;

Define the set of dimensional equations:

In [2]:
eqs_dim_clean = [
    -f_dim * v_dim ~ -(1 / ρ₀) * Dx(p_dim) + Dz(ν_dim * Dz(u_dim)),
     f_dim * u_dim ~ -(1 / ρ₀) * Dy(p_dim) + Dz(ν_dim * Dz(v_dim)),
     Dx(u_dim) + Dy(v_dim) + Dz(w_dim) ~ 0,
     Dz(p_dim) ~ ρ₀ * b_dim ,
     u_dim * Dx(b_dim) + v_dim * Dy(b_dim) + w_dim * Dz(b_dim) ~ Dz(κ_dim * Dz(b_dim)) + γ * (B_dim - b_dim)
]

5-element Vector{Equation}:
 -v(x, y, z)*f(y) ~ Differential(z, 1)(ν(z)*Differential(z, 1)(u(x, y, z))) + (-Differential(x, 1)(p(x, y, z))) / ρ₀
 u(x, y, z)*f(y) ~ Differential(z, 1)(ν(z)*Differential(z, 1)(v(x, y, z))) + (-Differential(y, 1)(p(x, y, z))) / ρ₀
 Differential(x, 1)(u(x, y, z)) + Differential(y, 1)(v(x, y, z)) + Differential(z, 1)(w(x, y, z)) ~ 0
 Differential(z, 1)(p(x, y, z)) ~ b(x, y, z)*ρ₀
 u(x, y, z)*Differential(x, 1)(b(x, y, z)) + Differential(z, 1)(b(x, y, z))*w(x, y, z) + v(x, y, z)*Differential(y, 1)(b(x, y, z)) ~ Differential(z, 1)(Differential(z, 1)(b(x, y, z))*κ(z)) + (B(x, y, z) - b(x, y, z))*γ

Define the scales for non-dimensionalization, and associated operators.

In [3]:
@variables L H0 f0 N0 κ0 ν0 τ0 γ U0

# Scales
Xscale = L
Yscale = L
Zscale = H0
Tscale = L / U0

Uscale = U0
Vscale = U0
Wscale = H0 * U0 / L

Pscale = ρ₀ * f0 * L * U0
bscale = N0^2 * H0
Bscale = bscale

fscale = f0
νscale = ν0
κscale = κ0
τscale = τ0

nd_subs = Dict(
    # Fields
    u_dim => Uscale * û,
    v_dim => Vscale * v̂,
    w_dim => Wscale * ŵ,
    p_dim => Pscale * p̂,
    b_dim => bscale * b̂,
    B_dim => Bscale * B̂,

    f_dim => fscale * f̂,
    ν_dim => νscale * ν̂,
    κ_dim => κscale * κ̂,

    # First derivatives: velocity
    Dx(u_dim) => Uscale / L  * Dx(û),
    Dy(u_dim) => Uscale / L  * Dy(û),
    Dz(u_dim) => Uscale / H0 * Dz(û),

    Dx(v_dim) => Vscale / L  * Dx(v̂),
    Dy(v_dim) => Vscale / L  * Dy(v̂),
    Dz(v_dim) => Vscale / H0 * Dz(v̂),

    Dx(w_dim) => Wscale / L  * Dx(ŵ),
    Dy(w_dim) => Wscale / L  * Dy(ŵ),
    Dz(w_dim) => Wscale / H0 * Dz(ŵ),

    # First derivatives: pressure
    Dx(p_dim) => Pscale / L  * Dx(p̂),
    Dy(p_dim) => Pscale / L  * Dy(p̂),
    Dz(p_dim) => Pscale / H0 * Dz(p̂),

    # First derivatives: buoyancy
    Dx(b_dim) => bscale / L  * Dx(b̂),
    Dy(b_dim) => bscale / L  * Dy(b̂),
    Dz(b_dim) => bscale / H0 * Dz(b̂),

    # Vertical viscosity operators
    Dz(ν_dim * Dz(u_dim)) =>
        νscale * Uscale / H0^2 * Dz(ν̂ * Dz(û)),

    Dz(ν_dim * Dz(v_dim)) =>
        νscale * Vscale / H0^2 * Dz(ν̂ * Dz(v̂)),

    # Vertical buoyancy diffusion operator
    Dz(κ_dim * Dz(b_dim)) =>
        κscale * bscale / H0^2 * Dz(κ̂ * Dz(b̂))
) ;

Make substitutions into dimensional equations

In [4]:
function substitute_equation(eq, subs)
    substitute(eq.lhs, subs) ~ substitute(eq.rhs, subs)
end

function simplify_equation(eq)
    simplify(eq.lhs) ~ simplify(eq.rhs)
end

eqs_scaled_by_substitution =
    simplify_equation.(substitute_equation.(eqs_dim_clean, Ref(nd_subs)))

5-element Vector{Equation}:
 -U0*f0*v_nd(x, y, z)*f_nd(y) ~ (U0*Differential(z, 1)(ν_nd(z)*Differential(z, 1)(u_nd(x, y, z)))*ν0 - (H0^2)*U0*f0*Differential(x, 1)(p_nd(x, y, z))) / (H0^2)
 U0*f0*u_nd(x, y, z)*f_nd(y) ~ (U0*Differential(z, 1)(Differential(z, 1)(v_nd(x, y, z))*ν_nd(z))*ν0 - (H0^2)*U0*f0*Differential(y, 1)(p_nd(x, y, z))) / (H0^2)
 (U0*Differential(y, 1)(v_nd(x, y, z)) + U0*Differential(z, 1)(w_nd(x, y, z)) + U0*Differential(x, 1)(u_nd(x, y, z))) / L ~ 0
 (L*U0*f0*Differential(z, 1)(p_nd(x, y, z))*ρ₀) / H0 ~ H0*(N0^2)*b_nd(x, y, z)*ρ₀
 (H0*(N0^2)*U0*v_nd(x, y, z)*Differential(y, 1)(b_nd(x, y, z)) + H0*(N0^2)*U0*Differential(z, 1)(b_nd(x, y, z))*w_nd(x, y, z) + H0*(N0^2)*U0*u_nd(x, y, z)*Differential(x, 1)(b_nd(x, y, z))) / L ~ ((N0^2)*Differential(z, 1)(Differential(z, 1)(b_nd(x, y, z))*κ_nd(z))*κ0 + H0*(H0*(N0^2)*B_nd(x, y, z) - H0*(N0^2)*b_nd(x, y, z))*γ) / H0

Manipulate scaled equations

In [5]:
mom_scale = f0 * U0
cont_scale = U0 / L
hydro_scale = Pscale / H0
buoy_adv_scale = U0 * bscale / L

eq_xmom_nd_raw =
    simplify(eqs_scaled_by_substitution[1].lhs / mom_scale ~
             eqs_scaled_by_substitution[1].rhs / mom_scale)

eq_ymom_nd_raw =
    simplify(eqs_scaled_by_substitution[2].lhs / mom_scale ~
             eqs_scaled_by_substitution[2].rhs / mom_scale)

eq_cont_nd_raw =
    simplify(eqs_scaled_by_substitution[3].lhs / cont_scale ~
             eqs_scaled_by_substitution[3].rhs / cont_scale)

eq_hydro_nd_raw =
    simplify(eqs_scaled_by_substitution[4].lhs / hydro_scale ~
             eqs_scaled_by_substitution[4].rhs / hydro_scale)

eq_buoy_nd_raw =
    simplify(eqs_scaled_by_substitution[5].lhs / buoy_adv_scale ~
             eqs_scaled_by_substitution[5].rhs / buoy_adv_scale)

eqs_nd_raw = [
    eq_xmom_nd_raw,
    eq_ymom_nd_raw,
    eq_cont_nd_raw,
    eq_hydro_nd_raw,
    eq_buoy_nd_raw
]

5-element Vector{Equation}:
 -v_nd(x, y, z)*f_nd(y) ~ (Differential(z, 1)(ν_nd(z)*Differential(z, 1)(u_nd(x, y, z)))*ν0 - (H0^2)*f0*Differential(x, 1)(p_nd(x, y, z))) / ((H0^2)*f0)
 u_nd(x, y, z)*f_nd(y) ~ (Differential(z, 1)(Differential(z, 1)(v_nd(x, y, z))*ν_nd(z))*ν0 - (H0^2)*f0*Differential(y, 1)(p_nd(x, y, z))) / ((H0^2)*f0)
 Differential(y, 1)(v_nd(x, y, z)) + Differential(z, 1)(w_nd(x, y, z)) + Differential(x, 1)(u_nd(x, y, z)) ~ 0
 Differential(z, 1)(p_nd(x, y, z)) ~ ((H0^2)*(N0^2)*b_nd(x, y, z)) / (L*U0*f0)
 v_nd(x, y, z)*Differential(y, 1)(b_nd(x, y, z)) + Differential(z, 1)(b_nd(x, y, z))*w_nd(x, y, z) + u_nd(x, y, z)*Differential(x, 1)(b_nd(x, y, z)) ~ (L*Differential(z, 1)(Differential(z, 1)(b_nd(x, y, z))*κ_nd(z))*κ0 + (H0^2)*L*B_nd(x, y, z)*γ - (H0^2)*L*b_nd(x, y, z)*γ) / ((H0^2)*U0)

Define non-dimensional numbers and substitute them:

In [6]:
@variables ϵ² Bu μ Γ

# Thermal-wind velocity scale:
# U0 = N0^2 H0^2 / (f0 L)
thermal_wind_subs = Dict(
    U0 => N0^2 * H0^2 / (f0 * L)
)

# Substitute dimensional parameters using nondimensional definitions.
# These substitutions are more robust than ratio-pattern replacement.
nd_param_subs_1 = Dict(
    ν0 => ϵ² * f0 * H0^2,           # Ekman number
    κ0 => ν0 / μ,                   # Prandtl number
    γ  => Γ * f0 / μ                # Relaxation number
)

nd_param_subs_2 = Dict(
    N0^2 => Bu * f0^2 * L^2 / H0^2, # Burger number
    ν0 => ϵ² * f0 * H0^2            # Ekman number 
)

# Extra identities implied by the nondimensional-number choices
closure_subs = Dict(
    U0 => Bu * L * f0
)

eqs_nd_step1 =
    simplify_equation.(substitute_equation.(eqs_nd_raw, Ref(thermal_wind_subs)))

eqs_nd_step2 =
    simplify_equation.(substitute_equation.(eqs_nd_step1, Ref(nd_param_subs_1)))

eqs_nd_step3 =
    simplify_equation.(substitute_equation.(eqs_nd_step2, Ref(nd_param_subs_2)))

5-element Vector{Equation}:
 -v_nd(x, y, z)*f_nd(y) ~ -Differential(x, 1)(p_nd(x, y, z)) + Differential(z, 1)(ν_nd(z)*Differential(z, 1)(u_nd(x, y, z)))*ϵ²
 u_nd(x, y, z)*f_nd(y) ~ -Differential(y, 1)(p_nd(x, y, z)) + Differential(z, 1)(Differential(z, 1)(v_nd(x, y, z))*ν_nd(z))*ϵ²
 Differential(y, 1)(v_nd(x, y, z)) + Differential(z, 1)(w_nd(x, y, z)) + Differential(x, 1)(u_nd(x, y, z)) ~ 0
 Differential(z, 1)(p_nd(x, y, z)) ~ b_nd(x, y, z)
 v_nd(x, y, z)*Differential(y, 1)(b_nd(x, y, z)) + Differential(z, 1)(b_nd(x, y, z))*w_nd(x, y, z) + u_nd(x, y, z)*Differential(x, 1)(b_nd(x, y, z)) ~ (Differential(z, 1)(Differential(z, 1)(b_nd(x, y, z))*κ_nd(z))*ϵ² + B_nd(x, y, z)*Γ - b_nd(x, y, z)*Γ) / (Bu*μ)

Helper functions to check the expressions:

In [7]:
clean_expr(expr) = simplify(expand(expr))

function clean_expr_with_closure(expr)
    clean_expr(substitute(expr, closure_subs))
end

clean_equation(eq) = clean_expr(eq.lhs) ~ clean_expr(eq.rhs)

function symbolic_difference(a, b; use_closure=true)
    diff = a - b

    if use_closure
        diff = substitute(diff, closure_subs)
    end

    clean_expr(diff)
end

function assert_expr_equal(a, b; msg="", use_closure=true)
    diff = symbolic_difference(a, b; use_closure=use_closure)

    @assert isequal(diff, 0) || string(diff) == "0" """
    $msg

    Expressions were not symbolically equal.

    Difference was:

    $diff
    """

    return true
end

function assert_equation_equal(eq, target; msg="", use_closure=true)
    assert_expr_equal(eq.lhs, target.lhs; msg="$msg\nLHS mismatch", use_closure=use_closure)
    assert_expr_equal(eq.rhs, target.rhs; msg="$msg\nRHS mismatch", use_closure=use_closure)
    return true
end ;

Check the final equation equals the equation in the text and display non-dimensional equations:

In [8]:
# Target equation, used only for assertion/checking
eq_buoy_nd_notes_target =
    μ * Bu * (û * Dx(b̂) + v̂ * Dy(b̂) + ŵ * Dz(b̂)) ~
    ϵ² * Dz(κ̂ * Dz(b̂)) + Γ * (B̂ - b̂)

# Derived buoyancy equation in the notes' final convention
eq_buoy_nd_notes_derived = clean_equation(
    (μ * Bu) * eqs_nd_step3[5].lhs ~
    (μ * Bu) * eqs_nd_step3[5].rhs
)

eqs_nd_notes_variable_coeffs = [
    clean_equation(eqs_nd_step3[1]),
    clean_equation(eqs_nd_step3[2]),
    clean_equation(eqs_nd_step3[3]),
    clean_equation(eqs_nd_step3[4]),
    eq_buoy_nd_notes_derived
]
display(eqs_nd_notes_variable_coeffs)

@assert assert_equation_equal(
    eq_buoy_nd_notes_derived,
    eq_buoy_nd_notes_target;
    msg="Derived buoyancy equation does not match the expected notes form."
)

println("Assertion passed: the derived buoyancy equation is symbolically equal to the target notes form.")
println("Therefore we may display the buoyancy equation in the cleaner target form:")

display(eq_buoy_nd_notes_target)


5-element Vector{Equation}:
 -v_nd(x, y, z)*f_nd(y) ~ -Differential(x, 1)(p_nd(x, y, z)) + Differential(z, 1)(ν_nd(z)*Differential(z, 1)(u_nd(x, y, z)))*ϵ²
 u_nd(x, y, z)*f_nd(y) ~ -Differential(y, 1)(p_nd(x, y, z)) + Differential(z, 1)(Differential(z, 1)(v_nd(x, y, z))*ν_nd(z))*ϵ²
 Differential(y, 1)(v_nd(x, y, z)) + Differential(z, 1)(w_nd(x, y, z)) + Differential(x, 1)(u_nd(x, y, z)) ~ 0
 Differential(z, 1)(p_nd(x, y, z)) ~ b_nd(x, y, z)
 Bu*v_nd(x, y, z)*Differential(y, 1)(b_nd(x, y, z))*μ + Bu*Differential(z, 1)(b_nd(x, y, z))*w_nd(x, y, z)*μ + Bu*u_nd(x, y, z)*Differential(x, 1)(b_nd(x, y, z))*μ ~ Differential(z, 1)(Differential(z, 1)(b_nd(x, y, z))*κ_nd(z))*ϵ² + B_nd(x, y, z)*Γ - b_nd(x, y, z)*Γ

Assertion passed: the derived buoyancy equation is symbolically equal to the target notes form.
Therefore we may display the buoyancy equation in the cleaner target form:


Bu*(v_nd(x, y, z)*Differential(y, 1)(b_nd(x, y, z)) + Differential(z, 1)(b_nd(x, y, z))*w_nd(x, y, z) + u_nd(x, y, z)*Differential(x, 1)(b_nd(x, y, z)))*μ ~ Differential(z, 1)(Differential(z, 1)(b_nd(x, y, z))*κ_nd(z))*ϵ² + (B_nd(x, y, z) - b_nd(x, y, z))*Γ

Check boundary conditions:

In [9]:
# New nondimensional/topographic variables and fields
@variables h_nd(..) bs_nd(..) τx_nd(..) τy_nd(..)

ĥ  = h_nd(x,y)
bŝ = bs_nd(x,y)
τx̂ = τx_nd(x,y)
τŷ = τy_nd(x,y)

# Dimensional bottom/topographic height and prescribed surface fields
@variables Htopo(..) bs(..) τx(..) τy(..)

H_dim  = Htopo(x,y)
bs_dim = bs(x,y)
τx_dim = τx(x,y)
τy_dim = τy(x,y)

# Aspect-ratio parameter for the slope term in the bottom buoyancy BC
@variables δ² 𝒯 ;

Boundary-condition scales and boundary-condition substitution dictionary


In [10]:
Hscale  = H0
bsscale = bscale
τscale  = τ0

bc_subs = Dict(
    # Dimensional boundary data
    H_dim  => Hscale * ĥ,
    bs_dim => bsscale * bŝ,
    τx_dim => τscale * τx̂,
    τy_dim => τscale * τŷ,

    # Horizontal derivatives of topography
    Dx(H_dim) => Hscale / L * Dx(ĥ),
    Dy(H_dim) => Hscale / L * Dy(ĥ),

    # Dimensional fields evaluated as scaled nondimensional fields
    u_dim => Uscale * û,
    v_dim => Vscale * v̂,
    w_dim => Wscale * ŵ,
    b_dim => bscale * b̂,

    ν_dim => νscale * ν̂,

    # Derivatives needed by boundary conditions
    Dz(u_dim) => Uscale / H0 * Dz(û),
    Dz(v_dim) => Vscale / H0 * Dz(v̂),

    Dx(b_dim) => bscale / L  * Dx(b̂),
    Dy(b_dim) => bscale / L  * Dy(b̂),
    Dz(b_dim) => bscale / H0 * Dz(b̂)
) ;

Helpers for boundary-condition equations

In [11]:
function substitute_bc(eq, subs)
    substitute(eq.lhs, subs) ~ substitute(eq.rhs, subs)
end

function scale_bc(eq, scale)
    clean_equation(eq.lhs / scale ~ eq.rhs / scale)
end

function apply_closure(eq; extra_subs=Dict())
    subs = merge(closure_subs, extra_subs)
    clean_equation(substitute(eq.lhs, subs) ~ substitute(eq.rhs, subs))
end ;

Dimensional surface stress boundary conditions

In [ ]:
bc_surf_stress_x_dim = ρ₀ * ν_dim * Dz(u_dim) ~ τx_dim
bc_surf_stress_y_dim = ρ₀ * ν_dim * Dz(v_dim) ~ τy_dim

# Substitute scalings
bc_surf_stress_x_scaled = substitute_bc(bc_surf_stress_x_dim, bc_subs)
bc_surf_stress_y_scaled = substitute_bc(bc_surf_stress_y_dim, bc_subs)

# Natural stress balance scale
stress_bc_scale = ρ₀ * ν0 * U0 / H0

bc_surf_stress_x_nd_raw = scale_bc(bc_surf_stress_x_scaled, stress_bc_scale)
bc_surf_stress_y_nd_raw = scale_bc(bc_surf_stress_y_scaled, stress_bc_scale)

# Use direct parameter substitution τ0 => ρ0 * ν0 * U0 / H0.
stress_param_subs = Dict(
    τ0 =>  stress_bc_scale
)

bc_surf_stress_x_nd = apply_closure(bc_surf_stress_x_nd_raw; extra_subs=stress_param_subs)
bc_surf_stress_y_nd = apply_closure(bc_surf_stress_y_nd_raw; extra_subs=stress_param_subs)

# Target nondimensional surface stress BCs
bc_surf_stress_x_target = ν̂ * Dz(û) ~  τx̂
bc_surf_stress_y_target = ν̂ * Dz(v̂) ~  τŷ

@assert assert_equation_equal(
    bc_surf_stress_x_nd,
    bc_surf_stress_x_target;
    msg="Surface x-stress BC scaling failed."
)

@assert assert_equation_equal(
    bc_surf_stress_y_nd,
    bc_surf_stress_y_target;
    msg="Surface y-stress BC scaling failed."
)

println("Assertion passed: surface stress BCs scale to")
[bc_surf_stress_x_target, bc_surf_stress_y_target]

Assertion passed: surface stress BCs scale to


2-element Vector{Equation}:
 ν_nd(z)*Differential(z, 1)(u_nd(x, y, z)) ~ τx_nd(x, y)
 Differential(z, 1)(v_nd(x, y, z))*ν_nd(z) ~ τy_nd(x, y)

Bottom no-slip boundary conditions

In [13]:
bc_bottom_noslip_u_dim = u_dim ~ 0
bc_bottom_noslip_v_dim = v_dim ~ 0

bc_bottom_noslip_u_scaled = substitute_bc(bc_bottom_noslip_u_dim, bc_subs)
bc_bottom_noslip_v_scaled = substitute_bc(bc_bottom_noslip_v_dim, bc_subs)

bc_bottom_noslip_u_nd = scale_bc(bc_bottom_noslip_u_scaled, Uscale)
bc_bottom_noslip_v_nd = scale_bc(bc_bottom_noslip_v_scaled, Vscale)

bc_bottom_noslip_u_target = û ~ 0
bc_bottom_noslip_v_target = v̂ ~ 0

@assert assert_equation_equal(
    bc_bottom_noslip_u_nd,
    bc_bottom_noslip_u_target;
    msg="Bottom no-slip u BC scaling failed."
)

@assert assert_equation_equal(
    bc_bottom_noslip_v_nd,
    bc_bottom_noslip_v_target;
    msg="Bottom no-slip v BC scaling failed."
)

println("Assertion passed: bottom no-slip BCs scale to")
[bc_bottom_noslip_u_target, bc_bottom_noslip_v_target]

Assertion passed: bottom no-slip BCs scale to


2-element Vector{Equation}:
 u_nd(x, y, z) ~ 0
 v_nd(x, y, z) ~ 0

Dimensional bottom location: z_dim = -H_dim.
In nondimensional variables: z = -ĥ.

In [14]:
bottom_location_scaled = substitute(H_dim, bc_subs) / H0
bottom_location_target = ĥ

@assert assert_expr_equal(
    bottom_location_scaled,
    bottom_location_target;
    msg="Bottom location scaling H/H0 = h failed."
)

println("Assertion passed: bottom location scales to z = -h(x,y).")

Assertion passed: bottom location scales to z = -h(x,y).


Surface buoyancy boundary condition

In [15]:
bc_surface_b_dim = b_dim ~ bs_dim

bc_surface_b_scaled = substitute_bc(bc_surface_b_dim, bc_subs)

bc_surface_b_nd = scale_bc(bc_surface_b_scaled, bscale)

bc_surface_b_target = b̂ ~ bŝ

@assert assert_equation_equal(
    bc_surface_b_nd,
    bc_surface_b_target;
    msg="Surface buoyancy BC scaling failed."
)

println("Assertion passed: surface buoyancy BC scales to")
bc_surface_b_target

Assertion passed: surface buoyancy BC scales to


b_nd(x, y, z) ~ bs_nd(x, y)

Bottom no-normal-buoyancy-gradient BC

In [16]:
bc_bottom_bnormal_dim =
    Dx(H_dim) * Dx(b_dim) +
    Dy(H_dim) * Dy(b_dim) +
    Dz(b_dim) ~ 0

bc_bottom_bnormal_scaled =
    substitute_bc(bc_bottom_bnormal_dim, bc_subs)

# Natural scale for b_z is bscale/H0 = N0^2
bnormal_bc_scale = bscale / H0

bc_bottom_bnormal_nd_raw =
    scale_bc(bc_bottom_bnormal_scaled, bnormal_bc_scale)

# Replace H0^2/L^2 by δ².
aspect_subs = Dict(
    H0^2 => δ² * L^2
)

bc_bottom_bnormal_nd =
    apply_closure(bc_bottom_bnormal_nd_raw; extra_subs=aspect_subs)

bc_bottom_bnormal_target =
    Dz(b̂) + δ² * (Dx(ĥ) * Dx(b̂) + Dy(ĥ) * Dy(b̂)) ~ 0

@assert assert_equation_equal(
    bc_bottom_bnormal_nd,
    bc_bottom_bnormal_target;
    msg="Bottom no-normal-buoyancy-gradient BC scaling failed."
)

println("Assertion passed: bottom no-normal-buoyancy-gradient BC scales to")
[bc_bottom_bnormal_target]

Assertion passed: bottom no-normal-buoyancy-gradient BC scales to


1-element Vector{Equation}:
 Differential(z, 1)(b_nd(x, y, z)) + (Differential(y, 1)(h_nd(x, y))*Differential(y, 1)(b_nd(x, y, z)) + Differential(x, 1)(h_nd(x, y))*Differential(x, 1)(b_nd(x, y, z)))*δ² ~ 0